In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

np.random.seed(42)
random.seed(42)
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def mean_relative_error(y_true, y_pred):
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mre = mean_relative_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mre': mre, 'r2': r2}

print("=" * 60)
print("Step 1/3: Data Preprocessing (CN Prediction)")
print("-" * 60)

print("Loading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

pca_image = PCA(n_components=0.95, random_state=42)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

label_data = pd.read_csv('label_r2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_cn = label_data[:, 1]
total_samples = len(label_cn)

unique_groups = np.unique(groups)
train_groups = unique_groups
train_mask = np.isin(groups, train_groups)
train_indices = np.where(train_mask)[0]
train_groups_labels = groups[train_mask]

param_space = {
    'n_estimators': [50, 100, 200, 300, 500, 800, 1000],
    'max_depth': [3, 4, 6, 8, 10, 12, 15, 20, None],
    'min_samples_split': [2, 4, 6, 8, 12, 16, 20, 25],
    'min_samples_leaf': [1, 2, 3, 4, 6, 8, 10, 15],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, None]
}

TRIALS_PER_MODEL = 30
GAP_THRESHOLD = 0.15

cv_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

print("\n" + "=" * 60)
print("Step 2/3: Hyperparameter Tuning (CN Random Search)")
print("-" * 60)

K_VALUES = [2, 3, 4, 5, 6, 7, 8]
ALL_RESULTS = []
ALL_PREDICTIONS = []
BEST_OVERALL = None

for k in K_VALUES:
    print(f"\nTesting K = {k}")
    
    selector = SelectKBest(score_func=f_regression, k=k)
    pattern_image_pca_selected = selector.fit_transform(pattern_image_pca_full, label_cn)
    selected_indices = selector.get_support(indices=True)
    
    selected_variance = image_pca_variance[selected_indices]
    cumulative_variance = np.sum(selected_variance)
    
    print(f"  Selected PC indices: {selected_indices}")
    print(f"  Cumulative variance ratio: {cumulative_variance:.4f}")
    print(f"  Running random search ({TRIALS_PER_MODEL} trials)...")
    
    X_train = pattern_image_pca_selected[train_mask]
    y_train = label_cn[train_mask]
    
    base_model = RandomForestRegressor(
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )
    
    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_space,
        n_iter=TRIALS_PER_MODEL,
        scoring='r2',
        cv=cv_splitter,
        random_state=42,
        n_jobs=-1,
        return_train_score=True,
        verbose=0
    )
    
    random_search.fit(X_train, y_train, groups=train_groups_labels)
    
    train_idx_in_train, val_idx_in_train = next(cv_splitter.split(X_train, y_train, groups=train_groups_labels))
    tr_idx = train_indices[train_idx_in_train]
    val_idx = train_indices[val_idx_in_train]
    
    best_params = random_search.best_params_
    best_model = random_search.best_estimator_
    
    y_tr_pred = best_model.predict(X_train[train_idx_in_train])
    y_val_pred = best_model.predict(X_train[val_idx_in_train])
    y_train_full_pred = best_model.predict(X_train)
    
    tr_metrics = calculate_metrics(y_train[train_idx_in_train], y_tr_pred)
    val_metrics = calculate_metrics(y_train[val_idx_in_train], y_val_pred)
    train_full_metrics = calculate_metrics(y_train, y_train_full_pred)
    
    train_val_gap = abs(train_full_metrics['r2'] - val_metrics['r2'])
    is_valid = train_val_gap < GAP_THRESHOLD
    
    all_pred = np.full(total_samples, np.nan)
    data_type = np.full(total_samples, "Unused")
    
    all_pred[tr_idx] = y_tr_pred
    all_pred[val_idx] = y_val_pred
    
    data_type[tr_idx] = "Training Set"
    data_type[val_idx] = "Validation Set"
    
    for idx in range(total_samples):
        ALL_PREDICTIONS.append({
            "Original_Index": idx + 1,
            "True_CN": round(label_cn[idx], 6),
            "Model_Type": f"ImageOnly_CN_K={k}",
            "Predicted_CN": round(all_pred[idx], 6) if not np.isnan(all_pred[idx]) else np.nan,
            "Data_Set_Type": data_type[idx]
        })
    
    result_entry = {
        'K': k,
        'Selected_Image_PCs': str(selected_indices),
        'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
        'Best_Params': str(best_params),
        'Train_R2': round(train_full_metrics['r2'], 6),
        'Train_MAE': round(train_full_metrics['mae'], 6),
        'Train_MSE': round(train_full_metrics['mse'], 6),
        'Train_MRE': round(train_full_metrics['mre'], 6),
        'Train_RMSE': round(train_full_metrics['rmse'], 6),
        'Val_R2': round(val_metrics['r2'], 6),
        'Val_MAE': round(val_metrics['mae'], 6),
        'Val_MSE': round(val_metrics['mse'], 6),
        'Val_MRE': round(val_metrics['mre'], 6),
        'Val_RMSE': round(val_metrics['rmse'], 6),
        'Train_Val_Gap': round(train_val_gap, 6),
        'Is_Valid': is_valid
    }
    
    ALL_RESULTS.append(result_entry)
    print(f"  Done | Train R²: {train_full_metrics['r2']:.6f} | Val R²: {val_metrics['r2']:.6f}")
    print(f"  Train-Val Gap: {train_val_gap:.6f} | Valid: {is_valid}")
    
    if BEST_OVERALL is None or val_metrics['r2'] > BEST_OVERALL['Val_R2']:
        BEST_OVERALL = result_entry

print("\n" + "=" * 60)
print("Step 3/3: CN Result Summary")
print("-" * 60)

results_df = pd.DataFrame(ALL_RESULTS)
results_df_sorted = results_df.sort_values(by='Val_R2', ascending=False).reset_index(drop=True)

predictions_df = pd.DataFrame(ALL_PREDICTIONS)
predictions_df = predictions_df.sort_values(by=['Original_Index', 'Model_Type']).reset_index(drop=True)

summary_cols = ['K', 'Cumulative_Variance_Ratio', 'Train_R2', 'Val_R2', 'Train_Val_Gap', 'Is_Valid']
print(results_df_sorted[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("Best CN Model Details")
print("-" * 60)
print(f"Optimal K: {BEST_OVERALL['K']}")
print(f"Selected PCs: {BEST_OVERALL['Selected_Image_PCs']}")
print(f"Cumulative variance: {BEST_OVERALL['Cumulative_Variance_Ratio']:.4f}")
print(f"Best parameters: {BEST_OVERALL['Best_Params']}")
print("\nFull Metrics:")
print(f"[Train] R²: {BEST_OVERALL['Train_R2']:.6f} | MAE: {BEST_OVERALL['Train_MAE']:.6f} | MSE: {BEST_OVERALL['Train_MSE']:.6f} | MRE: {BEST_OVERALL['Train_MRE']:.6f} | RMSE: {BEST_OVERALL['Train_RMSE']:.6f}")
print(f"[Val]   R²: {BEST_OVERALL['Val_R2']:.6f} | MAE: {BEST_OVERALL['Val_MAE']:.6f} | MSE: {BEST_OVERALL['Val_MSE']:.6f} | MRE: {BEST_OVERALL['Val_MRE']:.6f} | RMSE: {BEST_OVERALL['Val_RMSE']:.6f}")
print(f"\nTrain-Val Gap: {BEST_OVERALL['Train_Val_Gap']:.6f}")

with pd.ExcelWriter('ImageFeature_CN_RF_Tuning_Results.xlsx', engine='openpyxl') as writer:
    results_df_sorted.to_excel(writer, sheet_name='All_K_Results', index=False)
    pd.DataFrame([BEST_OVERALL]).to_excel(writer, sheet_name='Best_Model', index=False)
    predictions_df.to_excel(writer, sheet_name='Sample_Predictions', index=False)

print(f"\nResults saved to: ImageFeature_CN_RF_Tuning_Results.xlsx")
print("=" * 60)

Step 1/3: Data Preprocessing (CN Prediction)
------------------------------------------------------------
Loading image features...
Image PCA completed: 186 components, cumulative variance ratio: 0.9503
Top 5 PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Step 2/3: Hyperparameter Tuning (CN Random Search)
------------------------------------------------------------

Testing K = 2
  Selected PC indices: [6 8]
  Cumulative variance ratio: 0.0507
  Running random search (30 trials)...
  Done | Train R²: 0.221036 | Val R²: 0.160591
  Train-Val Gap: 0.060444 | Valid: True

Testing K = 3
  Selected PC indices: [4 6 8]
  Cumulative variance ratio: 0.0922
  Running random search (30 trials)...
  Done | Train R²: 0.251752 | Val R²: 0.213834
  Train-Val Gap: 0.037917 | Valid: True

Testing K = 4
  Selected PC indices: [ 4  6  8 10]
  Cumulative variance ratio: 0.1100
  Running random search (30 trials)...
  Done | Train R

In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import pandas as pd
import random

np.random.seed(42)
random.seed(42)

def build_reg_model(params):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

def mean_relative_error(y_true, y_pred):
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mre = mean_relative_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mre': mre, 'r2': r2}

print("=" * 60)
print("Step 1/4: Data Preprocessing")
print("-" * 60)

pattern = pd.read_csv('4-pattern2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
pattern = np.where(np.isinf(pattern), np.nan, pattern)
mean_val = np.nanmean(pattern) if not np.isnan(np.nanmean(pattern)) else 0
pattern = np.nan_to_num(pattern, nan=mean_val)

min_vals = np.min(pattern, axis=0)
max_vals = np.max(pattern, axis=0)
range_vals = np.where(max_vals - min_vals == 0, 1, max_vals - min_vals)
pattern_normalized = (pattern - min_vals) / range_vals

scaler_spectral = StandardScaler()
pattern_scaled = scaler_spectral.fit_transform(pattern_normalized)

print("\nLoading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

label_data = pd.read_csv('label_r2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_c = label_data[:, 1]
total_samples = len(label_c)

unique_groups = np.unique(groups)
test_size_groups = int(0.2 * len(unique_groups))
train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=test_size_groups,
    random_state=42
)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)
train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

X_train_raw_spectral = pattern_scaled[train_mask]
X_test_raw_spectral = pattern_scaled[test_mask]
y_train = label_c[train_mask]
y_test = label_c[test_mask]

pca_spectral = PCA(n_components=12)
X_train_spectral_pca = pca_spectral.fit_transform(X_train_raw_spectral)
X_test_spectral_pca = pca_spectral.transform(X_test_raw_spectral)

pca_image = PCA(n_components=30)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components retained, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 image PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

X_train_image_pca_full = pattern_image_pca_full[train_mask]
X_test_image_pca_full = pattern_image_pca_full[test_mask]

param_space = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

GAP_THRESHOLD = 0.15
CV_FOLDS = 5

print("\n" + "=" * 60)
print("Step 2/4: Pure Spectral Baseline Model")
print("-" * 60)

BASELINE_RESULT = {
    'Train_R2': 0.895846,
    'Train_MAE': 0.559925,
    'Train_MSE': 0.479508,
    'Train_MRE': 0.035417,
    'Train_RMSE': 0.692465,
    'Val_R2': 0.830243,
    'Val_MAE': 0.705349,
    'Val_MSE': 0.734932,
    'Val_MRE': 0.043924,
    'Val_RMSE': 0.857282,
    'Test_R2': 0.784473,
    'Test_MAE': 0.633086,
    'Test_MSE': 0.583228,
    'Test_MRE': 0.040072,
    'Test_RMSE': 0.763694,
    'Train_Val_Gap': round(0.895846 - 0.830243, 6),
    'Train_Test_Gap': round(0.895846 - 0.784473, 6),
    'Best_Params': "{'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}",
    'Is_Valid': True
}

print("Using specified pure spectral baseline metrics")
print("Pure Spectral Baseline Metrics")
print(f"[Train] R²: {BASELINE_RESULT['Train_R2']:.6f} | MAE: {BASELINE_RESULT['Train_MAE']:.6f} | MSE: {BASELINE_RESULT['Train_MSE']:.6f} | MRE: {BASELINE_RESULT['Train_MRE']:.6f} | RMSE: {BASELINE_RESULT['Train_RMSE']:.6f}")
print(f"[Val] R²: {BASELINE_RESULT['Val_R2']:.6f} | MAE: {BASELINE_RESULT['Val_MAE']:.6f} | MSE: {BASELINE_RESULT['Val_MSE']:.6f} | MRE: {BASELINE_RESULT['Val_MRE']:.6f} | RMSE: {BASELINE_RESULT['Val_RMSE']:.6f}")
print(f"[Test] R²: {BASELINE_RESULT['Test_R2']:.6f} | MAE: {BASELINE_RESULT['Test_MAE']:.6f} | MSE: {BASELINE_RESULT['Test_MSE']:.6f} | MRE: {BASELINE_RESULT['Test_MRE']:.6f} | RMSE: {BASELINE_RESULT['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BASELINE_RESULT['Train_Val_Gap']:.6f} | Train-Test Gap: {BASELINE_RESULT['Train_Test_Gap']:.6f} | Valid: {BASELINE_RESULT['Is_Valid']}")

print("\n" + "=" * 60)
print("Step 3/4: Hyperparameter Tuning for Different K Values")
print("-" * 60)

K_VALUES = [2, 3, 4, 5, 6, 7, 8]
ALL_RESULTS = []
ALL_PREDICTIONS = []
BEST_OVERALL = None

train_groups = groups[train_mask]
group_kfold = GroupKFold(n_splits=CV_FOLDS)

for k in K_VALUES:
    print(f"\nTesting K = {k}")
    
    selector = SelectKBest(score_func=f_regression, k=k)
    X_train_image_pca_selected = selector.fit_transform(X_train_image_pca_full, y_train)
    X_test_image_pca_selected = selector.transform(X_test_image_pca_full)
    selected_indices = selector.get_support(indices=True)
    
    selected_variance = image_pca_variance[selected_indices]
    cumulative_variance = np.sum(selected_variance)
    
    print(f"  Selected image PC indices: {selected_indices}")
    print(f"  Selected PC variance ratios: {[round(v, 4) for v in selected_variance]}")
    print(f"  Cumulative variance ratio: {cumulative_variance:.4f}")
    print("  Running grid search with group cross-validation...")
    
    X_train_fused = np.hstack([X_train_spectral_pca, X_train_image_pca_selected])
    X_test_fused = np.hstack([X_test_spectral_pca, X_test_image_pca_selected])
    
    base_model = RandomForestRegressor(bootstrap=True, random_state=42, n_jobs=-1)
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_space,
        cv=group_kfold,
        scoring='r2',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train_fused, y_train, groups=train_groups)
    best_params = grid_search.best_params_
    best_cv_score = grid_search.best_score_
    
    print(f"  Best CV R²: {best_cv_score:.6f}")
    print(f"  Best parameters: {best_params}")
    
    X_tr, X_val, y_tr, y_val, tr_idx, val_idx = train_test_split(
        X_train_fused, y_train, train_indices,
        test_size=0.2,
        random_state=42
    )
    
    final_model = build_reg_model(best_params)
    final_model.fit(X_tr, y_tr)
    
    y_tr_pred = final_model.predict(X_tr)
    y_val_pred = final_model.predict(X_val)
    y_train_pred = final_model.predict(X_train_fused)
    y_test_pred = final_model.predict(X_test_fused)
    
    tr_metrics = calculate_metrics(y_tr, y_tr_pred)
    val_metrics = calculate_metrics(y_val, y_val_pred)
    train_metrics = calculate_metrics(y_train, y_train_pred)
    test_metrics = calculate_metrics(y_test, y_test_pred)
    
    train_val_gap = abs(train_metrics['r2'] - val_metrics['r2'])
    train_test_gap = abs(train_metrics['r2'] - test_metrics['r2'])
    is_valid = (train_val_gap < GAP_THRESHOLD) and (train_test_gap < GAP_THRESHOLD)
    
    best_trial = {
        'params': best_params,
        'tr_r2': tr_metrics['r2'],
        'tr_mae': tr_metrics['mae'],
        'tr_mse': tr_metrics['mse'],
        'tr_mre': tr_metrics['mre'],
        'tr_rmse': tr_metrics['rmse'],
        'val_r2': val_metrics['r2'],
        'val_mae': val_metrics['mae'],
        'val_mse': val_metrics['mse'],
        'val_mre': val_metrics['mre'],
        'val_rmse': val_metrics['rmse'],
        'train_r2': train_metrics['r2'],
        'train_mae': train_metrics['mae'],
        'train_mse': train_metrics['mse'],
        'train_mre': train_metrics['mre'],
        'train_rmse': train_metrics['rmse'],
        'test_r2': test_metrics['r2'],
        'test_mae': test_metrics['mae'],
        'test_mse': test_metrics['mse'],
        'test_mre': test_metrics['mre'],
        'test_rmse': test_metrics['rmse'],
        'train_val_gap': train_val_gap,
        'train_test_gap': train_test_gap,
        'is_valid': is_valid
    }
    
    all_pred = np.full(total_samples, np.nan)
    data_type = np.full(total_samples, "Unused")
    
    all_pred[tr_idx] = y_tr_pred
    all_pred[val_idx] = y_val_pred
    all_pred[test_indices] = y_test_pred
    
    data_type[tr_idx] = "Training Set"
    data_type[val_idx] = "Validation Set"
    data_type[test_indices] = "Test Set"
    
    for idx in range(total_samples):
        ALL_PREDICTIONS.append({
            "Original_Index": idx + 1,
            "True_Label": round(label_c[idx], 6),
            "Model_Type": f"Fused_K={k}",
            "Predicted_Label": round(all_pred[idx], 6) if not np.isnan(all_pred[idx]) else np.nan,
            "Data_Set_Type": data_type[idx]
        })
    
    result_entry = {
        'K': k,
        'Selected_Image_PCs': str(selected_indices),
        'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
        'Best_Params': str(best_params),
        'CV_R2': round(best_cv_score, 6),
        'Train_R2': round(best_trial['train_r2'], 6),
        'Train_MAE': round(best_trial['train_mae'], 6),
        'Train_MSE': round(best_trial['train_mse'], 6),
        'Train_MRE': round(best_trial['train_mre'], 6),
        'Train_RMSE': round(best_trial['train_rmse'], 6),
        'Val_R2': round(best_trial['val_r2'], 6),
        'Val_MAE': round(best_trial['val_mae'], 6),
        'Val_MSE': round(best_trial['val_mse'], 6),
        'Val_MRE': round(best_trial['val_mre'], 6),
        'Val_RMSE': round(best_trial['val_rmse'], 6),
        'Test_R2': round(best_trial['test_r2'], 6),
        'Test_MAE': round(best_trial['test_mae'], 6),
        'Test_MSE': round(best_trial['test_mse'], 6),
        'Test_MRE': round(best_trial['test_mre'], 6),
        'Test_RMSE': round(best_trial['test_rmse'], 6),
        'Train_Val_Gap': round(best_trial['train_val_gap'], 6),
        'Train_Test_Gap': round(best_trial['train_test_gap'], 6),
        'Is_Valid': best_trial['is_valid']
    }
    
    ALL_RESULTS.append(result_entry)
    print(f"  Test completed | Test R²: {best_trial['test_r2']:.6f} | Train-Test Gap: {best_trial['train_test_gap']:.6f} | Valid: {best_trial['is_valid']}")
    
    if BEST_OVERALL is None or best_trial['test_r2'] > BEST_OVERALL['Test_R2']:
        BEST_OVERALL = result_entry

print("\n" + "=" * 60)
print("Step 4/4: Tuning Results Summary")
print("-" * 60)

results_df = pd.DataFrame(ALL_RESULTS)
results_df_sorted = results_df.sort_values(by='Test_R2', ascending=False).reset_index(drop=True)

predictions_df = pd.DataFrame(ALL_PREDICTIONS)
predictions_df = predictions_df.sort_values(by=['Original_Index', 'Model_Type']).reset_index(drop=True)

summary_cols = ['K', 'Cumulative_Variance_Ratio', 'CV_R2', 'Test_R2', 'Test_MAE', 'Test_RMSE', 'Train_Test_Gap', 'Is_Valid']
print(results_df_sorted[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("Best Overall Fused Model Results")
print("-" * 60)
print(f"Optimal K value: {BEST_OVERALL['K']}")
print(f"Selected image PC indices: {BEST_OVERALL['Selected_Image_PCs']}")
print(f"Cumulative variance ratio: {BEST_OVERALL['Cumulative_Variance_Ratio']:.4f}")
print(f"Optimal parameters: {BEST_OVERALL['Best_Params']}")
print("\nDetailed Performance Metrics:")
print(f"[Train] R²: {BEST_OVERALL['Train_R2']:.6f} | MAE: {BEST_OVERALL['Train_MAE']:.6f} | MSE: {BEST_OVERALL['Train_MSE']:.6f} | MRE: {BEST_OVERALL['Train_MRE']:.6f} | RMSE: {BEST_OVERALL['Train_RMSE']:.6f}")
print(f"[Val] R²: {BEST_OVERALL['Val_R2']:.6f} | MAE: {BEST_OVERALL['Val_MAE']:.6f} | MSE: {BEST_OVERALL['Val_MSE']:.6f} | MRE: {BEST_OVERALL['Val_MRE']:.6f} | RMSE: {BEST_OVERALL['Val_RMSE']:.6f}")
print(f"[Test] R²: {BEST_OVERALL['Test_R2']:.6f} | MAE: {BEST_OVERALL['Test_MAE']:.6f} | MSE: {BEST_OVERALL['Test_MSE']:.6f} | MRE: {BEST_OVERALL['Test_MRE']:.6f} | RMSE: {BEST_OVERALL['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BEST_OVERALL['Train_Val_Gap']:.6f} | Train-Test Gap: {BEST_OVERALL['Train_Test_Gap']:.6f} | Valid: {BEST_OVERALL['Is_Valid']}")

train_improve_r2 = (BEST_OVERALL['Train_R2'] - BASELINE_RESULT['Train_R2']) * 100
train_improve_mae = (BASELINE_RESULT['Train_MAE'] - BEST_OVERALL['Train_MAE']) * 100
train_improve_mse = (BASELINE_RESULT['Train_MSE'] - BEST_OVERALL['Train_MSE']) * 100
train_improve_mre = (BASELINE_RESULT['Train_MRE'] - BEST_OVERALL['Train_MRE']) * 100
train_improve_rmse = (BASELINE_RESULT['Train_RMSE'] - BEST_OVERALL['Train_RMSE']) * 100

val_improve_r2 = (BEST_OVERALL['Val_R2'] - BASELINE_RESULT['Val_R2']) * 100
val_improve_mae = (BASELINE_RESULT['Val_MAE'] - BEST_OVERALL['Val_MAE']) * 100
val_improve_mse = (BASELINE_RESULT['Val_MSE'] - BEST_OVERALL['Val_MSE']) * 100
val_improve_mre = (BASELINE_RESULT['Val_MRE'] - BEST_OVERALL['Val_MRE']) * 100
val_improve_rmse = (BASELINE_RESULT['Val_RMSE'] - BEST_OVERALL['Val_RMSE']) * 100

test_improve_r2 = (BEST_OVERALL['Test_R2'] - BASELINE_RESULT['Test_R2']) * 100
test_improve_mae = (BASELINE_RESULT['Test_MAE'] - BEST_OVERALL['Test_MAE']) * 100
test_improve_mse = (BASELINE_RESULT['Test_MSE'] - BEST_OVERALL['Test_MSE']) * 100
test_improve_mre = (BASELINE_RESULT['Test_MRE'] - BEST_OVERALL['Test_MRE']) * 100
test_improve_rmse = (BASELINE_RESULT['Test_RMSE'] - BEST_OVERALL['Test_RMSE']) * 100

print("\nComparison with Pure Spectral Baseline (All Datasets):")
print("-"*50)
print(f"【Training Set】")
print(f"Baseline Train R²: {BASELINE_RESULT['Train_R2']:.6f} | Best Fused Train R²: {BEST_OVERALL['Train_R2']:.6f}")
print(f"Improvement: R² {train_improve_r2:+.2f}% | MAE {train_improve_mae:+.2f}% | MSE {train_improve_mse:+.2f}% | MRE {train_improve_mre:+.2f}% | RMSE {train_improve_rmse:+.2f}%")

print(f"\n【Validation Set】")
print(f"Baseline Val R²: {BASELINE_RESULT['Val_R2']:.6f} | Best Fused Val R²: {BEST_OVERALL['Val_R2']:.6f}")
print(f"Improvement: R² {val_improve_r2:+.2f}% | MAE {val_improve_mae:+.2f}% | MSE {val_improve_mse:+.2f}% | MRE {val_improve_mre:+.2f}% | RMSE {val_improve_rmse:+.2f}%")

print(f"\n【Test Set】")
print(f"Baseline Test R²: {BASELINE_RESULT['Test_R2']:.6f} | Best Fused Test R²: {BEST_OVERALL['Test_R2']:.6f}")
print(f"Improvement: R² {test_improve_r2:+.2f}% | MAE {test_improve_mae:+.2f}% | MSE {test_improve_mse:+.2f}% | MRE {test_improve_mre:+.2f}% | RMSE {test_improve_rmse:+.2f}%")

with pd.ExcelWriter('Image_Feature_K_Selection_Results_CN_Tuned.xlsx', engine='openpyxl') as writer:
    pd.DataFrame([BASELINE_RESULT]).to_excel(writer, sheet_name='Spectral_Baseline', index=False)
    results_df_sorted.to_excel(writer, sheet_name='All_K_Results', index=False)
    pd.DataFrame([BEST_OVERALL]).to_excel(writer, sheet_name='Best_Fused_Result', index=False)
    predictions_df.to_excel(writer, sheet_name='All_Sample_Predictions', index=False)

print(f"\nAll results saved to: Image_Feature_K_Selection_Results_CN_Tuned.xlsx")
print("=" * 60)

Step 1/4: Data Preprocessing
------------------------------------------------------------

Loading image features...
Image PCA completed: 30 components retained, cumulative variance ratio: 0.7159
Top 5 image PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Step 2/4: Pure Spectral Baseline Model
------------------------------------------------------------
Using specified pure spectral baseline metrics
Pure Spectral Baseline Metrics
[Train] R²: 0.895846 | MAE: 0.559925 | MSE: 0.479508 | MRE: 0.035417 | RMSE: 0.692465
[Val] R²: 0.830243 | MAE: 0.705349 | MSE: 0.734932 | MRE: 0.043924 | RMSE: 0.857282
[Test] R²: 0.784473 | MAE: 0.633086 | MSE: 0.583228 | MRE: 0.040072 | RMSE: 0.763694
Train-Val Gap: 0.065603 | Train-Test Gap: 0.111373 | Valid: True

Step 3/4: Hyperparameter Tuning for Different K Values
------------------------------------------------------------

Testing K = 2
  Selected image PC indices: [6 8]
  Sel